In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from skimage import exposure
import matplotlib.pyplot as plt

In [ ]:
img1 = cv2.imread('s1.jpg')
img2 = cv2.imread('s2.jpg')
cv2_imshow(cv2.hconcat([img1, img2]))

In [ ]:
# Conversione in scala di grigi
gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

# Estrazione dei keypoints e dei descrittori con SIFT
sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

cv2_imshow(cv2.hconcat([cv2.drawKeypoints(gray1, kp1, None), cv2.drawKeypoints(gray2, kp2, None)]))

In [ ]:
# Corrispondenza dei descrittori usando BFMatcher con L2 norm
bf = cv2.BFMatcher_create(cv2.NORM_L2, crossCheck=False)
matches = bf.knnMatch(des1, des2, k=2)

# Flann Matcher che sa uso di Kd-Tree per una ricerca più efficiente (non esaustiva)
#fm = cv2.FlannBasedMatcher_create()
#matches = fm.knnMatch(des1, des2, k=2)
print(matches[0])

# Usa il criterio di Lowe per filtrare i match
good_matches = []
for m, n in matches:
    if m.distance < 0.7 * n.distance:
        good_matches.append(m)

# Disegna i match sulle immagini
img_matches = cv2.drawMatches(img1, kp1, img2, kp2, good_matches[:50], None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
cv2_imshow(img_matches)

In [ ]:
# Controllo sul numero di corrispondenze trovate
if len(good_matches) > 5:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)

    # Calcola la matrice di omografia utilizzando RANSAC
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 1.0)

    # Calcola i bordi dell'immagine trasformata
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    corners_img1 = np.float32([[0, 0], [0, h1], [w1, h1], [w1, 0]]).reshape(-1, 1, 2)
    corners_img1_transformed = cv2.perspectiveTransform(corners_img1, H)
    corners = np.concatenate((corners_img1_transformed, np.float32([[0, 0], [0, h2], [w2, h2], [w2, 0]]).reshape(-1, 1, 2)), axis=0)
    #print(corners)

    # Trova i limiti del panorama
    [xmin, ymin] = np.int32(corners.min(axis=0).ravel() - 0.5)
    [xmax, ymax] = np.int32(corners.max(axis=0).ravel() + 0.5)

    print([xmin, ymin], [xmax, ymax])

    # Traslazione necessaria per evitare coordinate negative
    translation = [-xmin, -ymin]
    H_translation = np.array([[1, 0, translation[0]], [0, 1, translation[1]], [0, 0, 1]])

    # Allinea img1 e img2 nello spazio della panoramica
    panorama = cv2.warpPerspective(img1, H_translation.dot(H), (xmax - xmin, ymax - ymin))
    panorama[translation[1]:h2 + translation[1], translation[0]:w2 + translation[0]] = img2
    cv2_imshow(panorama)

In [ ]:
# Uso dello stitcher di opencv
stitcher = cv2.Stitcher.create()
(status, stitched) = stitcher.stitch([img1, img2])
if status == 0:
    cv2_imshow(stitched)
else:
    print("Stitching unsuccessful")